# OPENdraw · Milestone 4 — DCM Training Pipeline

Turns 200 Western comic pages (DCM-13k) into a validated LineScout gallery:
line-art extraction → MobileCLIP2 + DINOv2 embeddings → FAISS index → ready to serve.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (cell 1)
3. Upload `data/sources/dcm/` from your local machine to Drive (see cell 1)
4. Run cells top to bottom — each cell prints what it did before you continue

**Estimated time:** ~35–55 min for 200 pages on a free T4.

---
## Cell 1 · Mount Drive + configure paths

**Before running this cell**, upload your local `data/sources/dcm/` folder to Drive:
```
MyDrive/
  LineScout/
    sources/
      dcm/
        pages/          ← the 200 .jpg files
        source_registry.json
```
You can drag-and-drop the folder in drive.google.com, or use the Colab file browser.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Edit these if your Drive layout differs ───────────────────────────
DATASET_VERSION = '2026.10.03-dcm-pilot'   # change date for a fresh run
DRIVE_ROOT      = Path('/content/drive/MyDrive/LineScout')
DCM_ROOT        = DRIVE_ROOT / 'sources' / 'dcm' / 'pages'
OUTPUT_ROOT     = DRIVE_ROOT / 'gallery' / DATASET_VERSION
# ─────────────────────────────────────────────────────────────────────

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Verify sources landed
dcm_images = sorted(DCM_ROOT.glob('*.jpg'))
print(f'DCM pages found: {len(dcm_images)}')
if not dcm_images:
    raise FileNotFoundError(
        f'No .jpg files found at {DCM_ROOT}\n'
        'Upload data/sources/dcm/pages/ to MyDrive/LineScout/sources/dcm/pages/ first.'
    )

print(f'Gallery output  : {OUTPUT_ROOT}')
print(f'Sample filenames: {[p.name for p in dcm_images[:4]]}')

---
## Cell 2 · GPU check

**Expected output:**
```
GPU : Tesla T4  |  15.8 GB VRAM
CUDA: 12.x   PyTorch: 2.x.x+cu...
```
If you see `No GPU` go to `Runtime → Change runtime type → T4 GPU` and reconnect.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1024**3
print(f'GPU : {gpu.name}  |  {vram:.1f} GB VRAM')
print(f'CUDA: {torch.version.cuda}   PyTorch: {torch.__version__}')

if '+cu' not in torch.__version__:
    raise RuntimeError(
        f'torch {torch.__version__} is a CPU build — CUDA is missing.\n'
        'Reconnect with a T4 runtime; pip must not reinstall torch.'
    )
if vram < 12:
    print(f'WARNING: only {vram:.1f} GB VRAM — reduce BATCH_SIZE below if you get OOM')

---
## Cell 3 · Clone repo + install dependencies

Clones `linescout_ml` at the pinned commit, then installs the pinned Colab
requirements without touching Colab's CUDA-enabled torch.

**Takes:** ~3–5 min on first run. Subsequent runs reuse the clone.

**Expected output ends with:**
```
HEAD : 81a8683a53ea  (matches pin)
open_clip ✓   timm ✓   controlnet_aux ✓
Ready.
```

In [ ]:
import subprocess, sys, shutil
import torch, torchvision
from pathlib import Path

# ── Updated to point at the real repo and current HEAD pin ────────────
REPO_URL = 'https://github.com/AnweshBhattacharya/OPENdraw.git'
REPO_PIN = 'f40009e36900ae359cf518ea90310aca0425b07f'
REPO_DIR = Path('/content/OPENdraw')

# ── Clone and checkout the exact pinned commit ───────────────────────
# --depth 1 only clones main; we need a full clone to checkout any commit.
if REPO_DIR.exists():
    head = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
        capture_output=True, text=True
    ).stdout.strip()
    if head.startswith(REPO_PIN[:12]):
        print(f'Reusing existing clone at pin {REPO_PIN[:12]}')
    else:
        print(f'Existing clone is at {head[:12]}, re-cloning to get pin {REPO_PIN[:12]} …')
        shutil.rmtree(str(REPO_DIR))
        REPO_DIR = Path('/content/OPENdraw')  # reassign after rmtree

if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} …')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    print(f'Checking out pin {REPO_PIN[:12]} …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_PIN], check=True)

head = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    capture_output=True, text=True
).stdout.strip()
match = '(matches pin)' if head.startswith(REPO_PIN[:12]) else f'WARNING: expected {REPO_PIN[:12]}'
print(f'HEAD : {head[:12]}  {match}')

ml_path = str(REPO_DIR / 'ml')
if ml_path not in sys.path:
    sys.path.insert(0, ml_path)

# ── Install pinned deps (never reinstalls torch) ──────────────────────
REQS = REPO_DIR / 'ml' / 'colab' / 'requirements-colab.txt'
if not REQS.exists():
    raise FileNotFoundError(f'requirements-colab.txt not found at {REQS}\nCheck the repo clone.')

pins = []
in_runtime = False
for line in REQS.read_text().splitlines():
    stripped = line.strip()
    if stripped.startswith('# group: runtime'): in_runtime = True; continue
    if stripped.startswith('# group:'): in_runtime = False; continue
    if not in_runtime and '==' in stripped and not stripped.startswith('#'):
        pins.append(stripped)

print(f'Installing {len(pins)} pinned packages …')

# Freeze Colab's CUDA torch so pip cannot replace it
constraints = Path('/tmp/torch_freeze.txt')
constraints.write_text(f'torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n')

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '--constraint', str(constraints)] + pins,
    capture_output=True, text=True
)
if r.returncode != 0:
    print(r.stdout[-2000:]); print(r.stderr[-2000:])
    raise RuntimeError('pip install failed')

# Quick smoke check on the three heavy dependencies
for mod, label in [('open_clip', 'open_clip'), ('timm', 'timm'), ('controlnet_aux', 'controlnet_aux')]:
    try:
        __import__(mod); print(f'  {label} ✓')
    except ImportError:
        print(f'  {label} ✗  (import failed)')

# Confirm torch still has its CUDA build tag
assert '+cu' in torch.__version__, f'torch lost its CUDA build: {torch.__version__}'
print('Ready.')

---
## Cell 4 · Build pipeline config

Defines the DCM source with its licence terms and selects the
`informative_drawings` extractor (better for Western ink cross-hatching
than `anime2sketch`).

Adjust `BATCH_SIZE` here if you get OOM errors in later cells:
- T4 (16 GB): 8 is safe for both encoders simultaneously
- Shared/low-VRAM runtime: drop to 4

In [ ]:
from linescout_ml.colab.config import PipelineConfig, SourceSpec
from linescout_ml.taxonomy import PrimaryStyle, ScopeLabel, LineArtOrigin
from linescout_ml.taxonomy import PermissionBasis as PB

BATCH_SIZE = 8   # safe for T4 (16 GB); drop to 4 if OOM

dcm_source = SourceSpec(
    name               = 'dcm',
    root               = DCM_ROOT,
    license_id         = 'dcm-public-domain',
    permission_basis   = PB.PUBLIC_DOMAIN,
    allowed_display    = True,
    allowed_training   = True,
    allowed_trace      = False,   # conservative — upgrade per-asset in curation
    # informative_drawings preserves ink texture better than anime2sketch
    # for pre-1950s American comic art (cross-hatching, brushwork)
    extractor          = 'informative_drawings',
    default_style      = PrimaryStyle.WESTERN_INK,
    default_scopes     = [ScopeLabel.FULL_BODY],
    sfw_method         = 'source_rating',  # DCM is a curated public-domain archive
    work_grouping      = 'filename',       # one page = one work
)

config = PipelineConfig(
    dataset_version  = DATASET_VERSION,
    output_root      = OUTPUT_ROOT,
    sources          = [dcm_source],
    extract_line_art = True,
    label            = True,
    embed            = True,
    dedupe           = True,
    device           = 'auto',
    batch_size       = BATCH_SIZE,
)

print(f'PipelineConfig OK')
print(f'  dataset_version : {config.dataset_version}')
print(f'  source          : dcm  ({len(dcm_images)} images)')
print(f'  extractor       : {dcm_source.extractor}')
print(f'  embedders       : {config.embedders}')
print(f'  batch_size      : {config.batch_size}')
print(f'  output          : {config.output_root}')

---
## Cell 5 · Stage 1: Discover images

Assigns a deterministic `asset_id` and learning split to every image.
CPU-only, ~10 seconds.

**Expected output:**
```
Discovered 200 candidates
Splits  →  train: 140  validation: 30  test: 30
```

In [ ]:
from linescout_ml.colab.runner import PipelineRunner, candidate_summary
from tqdm.auto import tqdm

def _progress(stage, done, total):
    if done == 0 or done == total or done % 20 == 0:
        print(f'  {stage}: {done}/{total}', flush=True)

runner = PipelineRunner(config, progress=_progress)

print('Stage 1/6: Discovering …')
runner.discover()

candidates = list(runner.store)
splits = {}
for c in candidates:
    splits[c.learning_split] = splits.get(c.learning_split, 0) + 1

print(f'Discovered {len(candidates)} candidates')
print('Splits  →  ' + '  '.join(f'{k}: {v}' for k, v in sorted(splits.items())))

---
## Cell 6 · Stage 2: Extract line art  ⚡ GPU

Runs `informative_drawings` on every DCM page. Downloads model weights
(~60 MB) on first run.

Saves `originals/`, `line_art/`, `thumbnails/` to Drive. Resumable.

**Estimated time:** ~10–20 min for 200 pages on T4.

**Expected output ends with:**
```
  extract: 200/200
Extracted  done=200  skipped=0  failed=0
```

In [ ]:
print('Stage 2/6: Extracting line art (GPU) …')
result = runner.run_extract()
print(f'Extracted  done={result.processed}  skipped={result.skipped}  failed={result.failed}')
if result.failed > 0:
    print(f'  {result.failed} failed — check notes:')
    for note in result.notes[:10]: print(f'    {note}')

---
## Cell 7 · Stage 3: Measure + deduplicate

CPU-only. Computes ink coverage, text coverage, quality score, pHash.
Near-duplicate pages (Hamming ≤ 6) are grouped — only one enters the gallery.

**Expected output:**
```
Measured 200/200
Deduplication: removed=0  unique=200
```

In [ ]:
print('Stage 3/6: Measuring …')
m_result = runner.run_measure()
print(f'Measured {m_result.processed}/{m_result.processed + m_result.skipped}')

print('Stage 4/6: Deduplicating …')
d_result = runner.run_dedupe()
total = d_result.processed + d_result.skipped
print(f'Deduplication: removed={d_result.skipped}  unique={d_result.processed}')

---
## Cell 8 · Stage 4: Zero-shot labelling  ⚡ GPU

Uses MobileCLIP2-S2 to assign provisional `primary_scope` and `primary_style`
labels via text-image matching, then runs SFW screening.

Labels are **provisional** — the `/curate` UI lets you correct them.

**Estimated time:** ~4–8 min for 200 images on T4.

**Expected output ends with:**
```
Labelled 200/200
Top scopes: full_body=72  upper_body_clothing=48  face_head=36 …
SFW: safe=200  quarantined=0
```

In [ ]:
from collections import Counter

print('Stage 5/6: Labelling (GPU) …')
l_result = runner.run_label()
print(f'Labelled {l_result.processed}/{l_result.processed + l_result.skipped}')

# Show scope distribution
scope_counts = Counter(
    c.labels.primary_scope for c in runner.store if c.labels is not None
)
top = scope_counts.most_common(6)
print('Top scopes: ' + '  '.join(f'{s}={n}' for s, n in top))

# SFW summary
safe = sum(1 for c in runner.store if c.sfw_screening and c.sfw_screening.verdict == 'safe')
quarantined = sum(1 for c in runner.store if c.sfw_screening and c.sfw_screening.verdict != 'safe')
print(f'SFW: safe={safe}  quarantined={quarantined}')

---
## Cell 9 · Stage 5: Embed  ⚡ GPU

Runs both encoders on every line-art image:
- **MobileCLIP2-S2** → 512-d semantic embedding
- **DINOv2-small** → 384-d structural embedding

Writes resumable shard `.npz` files to `OUTPUT_ROOT/../indexes/`.

**Estimated time:** ~8–15 min for 200 images on T4.

**Expected output ends with:**
```
  mobileclip2_s2: 200/200  (512d)
  dinov2_vits14 : 200/200  (384d)
Embeddings written.
```

In [ ]:
print('Stage 6/6: Embedding (GPU) …')
e_result = runner.run_embed()
for enc, info in e_result.items():
    print(f'  {enc}: {info.processed}/{info.processed + info.skipped}  ({info.dim}d)')
print('Embeddings written.')

---
## Cell 10 · Build manifest + validate

Assembles a schema v3 `manifest.json` and validates it with the same
validator the API uses. All assets arrive `unreviewed` — that is expected.

**Expected output:**
```
Manifest written: 200 records  0 servable  (unreviewed — needs curation)
Validation: OK  content_hash=xxxxxxxxxxxx
```

In [ ]:
b_result = runner.run_build()
manifest_path = OUTPUT_ROOT / 'manifest.json'
print(f'Manifest written: {b_result.processed} records  '
      f'{b_result.skipped} servable  (unreviewed — needs curation)')

# Validate with the same validator the API uses
from linescout_ml.manifest import Manifest
from pydantic import ValidationError
try:
    m = Manifest.model_validate_json(manifest_path.read_text())
    print(f'Validation: OK  content_hash={m.content_hash()[:12]}')
except ValidationError as exc:
    print(f'Validation FAILED: {exc}')
    raise

---
## Cell 11 · Build FAISS index

Loads all embedding shards and builds two `IndexFlatIP` FAISS indexes
(exact cosine search, no approximation needed at 200 vectors).

Writes `faiss_mobileclip2_s2.index`, `faiss_dinov2_vits14.index`, and
`id_map.json` alongside the embedding shards.

**Expected output:**
```
mobileclip2_s2 : 200 vectors  dim=512  → …/mobileclip2_s2/faiss.index
dinov2_vits14  : 200 vectors  dim=384  → …/dinov2_vits14/faiss.index
```

In [ ]:
import json
import numpy as np

try:
    import faiss
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu==1.8.0'], check=True)
    import faiss

# Index root is next to the gallery root
indexes_root = OUTPUT_ROOT.parent.parent / 'indexes' / DATASET_VERSION

def build_faiss_index(enc_key: str) -> None:
    enc_dir = indexes_root / enc_key
    shards  = sorted(enc_dir.glob('shard-*.npz'))
    if not shards:
        print(f'  {enc_key}: no shards found — skipping')
        return

    meta      = json.loads((enc_dir / 'index.json').read_text())
    asset_ids = list(meta['entries'].keys())

    vec_map = {}
    for shard in shards:
        data = np.load(shard)
        for key in data.files:
            vec_map[key] = data[key]

    vectors = np.stack([vec_map[aid] for aid in asset_ids]).astype(np.float32)
    norms   = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors /= np.maximum(norms, 1e-8)

    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors)

    out_path = enc_dir / 'faiss.index'
    faiss.write_index(idx, str(out_path))
    (enc_dir / 'id_map.json').write_text(
        json.dumps({'encoder': enc_key, 'asset_ids': asset_ids}, indent=2)
    )
    print(f'  {enc_key}: {len(asset_ids)} vectors  dim={vectors.shape[1]}  → {out_path}')

print('Building FAISS indexes …')
build_faiss_index('mobileclip2_s2')
build_faiss_index('dinov2_vits14')

---
## Cell 12 · Export + download

Creates:
- `gallery.zip` — manifest + line_art + thumbnails (no originals, ~30–50 MB)
- `indexes.zip` — FAISS indexes + id maps (~5–10 MB)

Both are saved to Drive **and** downloaded to your local machine.

**Do not close the browser tab until both downloads finish.**

In [ ]:
import time, zipfile
from google.colab import files as colab_files

def zip_tree(src_root: Path, zip_path: Path, patterns: list[str]) -> int:
    count = 0
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for pattern in patterns:
            for path in sorted(src_root.glob(pattern)):
                if path.is_file():
                    zf.write(path, path.relative_to(src_root))
                    count += 1
    return count

# ── Gallery ZIP (line-art + thumbnails + manifest) ────────────────────
gallery_zip = Path('/content/gallery.zip')
n = zip_tree(OUTPUT_ROOT, gallery_zip,
             ['manifest.json', 'line_art/**/*.png', 'thumbnails/**/*.png'])
print(f'gallery.zip : {n} files  ({gallery_zip.stat().st_size/1024**2:.1f} MB)')

# ── Indexes ZIP ───────────────────────────────────────────────────────
indexes_zip = Path('/content/indexes.zip')
n2 = zip_tree(indexes_root, indexes_zip, ['**/*'])
print(f'indexes.zip : {n2} files  ({indexes_zip.stat().st_size/1024**2:.1f} MB)')

# ── Copy to Drive for safe-keeping ───────────────────────────────────
import shutil
drive_exports = DRIVE_ROOT / 'exports' / DATASET_VERSION
drive_exports.mkdir(parents=True, exist_ok=True)
shutil.copy(gallery_zip, drive_exports / 'gallery.zip')
shutil.copy(indexes_zip, drive_exports / 'indexes.zip')
print(f'Copied to Drive: {drive_exports}')

# ── Download to local machine ─────────────────────────────────────────
print('Starting downloads to your machine …')
colab_files.download(str(gallery_zip))
colab_files.download(str(indexes_zip))

---
## Cell 13 · Final summary

Shows record counts and reminds you what to do locally after unzipping.

In [ ]:
import json

manifest = json.loads((OUTPUT_ROOT / 'manifest.json').read_text())
records  = manifest.get('records', [])

print('='*55)
print(f'Dataset version : {DATASET_VERSION}')
print(f'Total records   : {len(records)}')
print(f'Servable now    : 0  (all unreviewed — expected)')
print('='*55)
print()
print('Next steps on your local machine:')
print()
print('  1. Unpack:')
print(f'     mkdir -p data/gallery/{DATASET_VERSION}')
print(f'     Expand-Archive gallery.zip -DestinationPath data/gallery/{DATASET_VERSION}/')
print(f'     mkdir -p data/indexes')
print(f'     Expand-Archive indexes.zip -DestinationPath data/indexes/')
print()
print('  2. Validate:')
print(f'     ml\\.venv\\Scripts\\python -m linescout_ml.cli validate \\')
print(f'       data/gallery/{DATASET_VERSION}/manifest.json --require-files')
print()
print('  3. Edit services/api/.env:')
print(f'     LINESCOUT_GALLERY_MANIFEST=data/gallery/{DATASET_VERSION}/manifest.json')
print(f'     LINESCOUT_CURATION_MODE=1')
print(f'     LINESCOUT_FIXTURE_MODE=false')
print()
print('  4. Start the app and curate:')
print('     npm run dev:all')
print('     → http://127.0.0.1:5173/curate')